# S0 — 02.07: Elastic Net

**Question:** How can one regression model combine L1 sparsity and L2 shrinkage, and how do the two penalty weights change its coefficients?

This is deterministic arithmetic on five synthetic, dimensionless observations. It is not evidence about accuracy, superiority, causality, feature importance, or generalisation. Gradient descent, preprocessing, cross-validation, inference, and deployment are outside this lesson.

## S1 — Samples, features, and shapes

`X.shape == (5, 2)` means five samples and two features. `y.shape == (5,)` has one target per sample. The parameter vector is `theta = [b, w1, w2]` with shape `(3,)`; the intercept `b` is not penalised.

In [ ]:
import numpy as np

X = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [1.0, 1.0],
              [2.0, 1.0],
              [1.0, 2.0]], dtype=np.float64)
y = np.array([5.0, 1.0, 4.0, 8.0, 3.0], dtype=np.float64)
print(f'X shape: {X.shape}, y shape: {y.shape}, dtype: {X.dtype}')
print('x means:', X.mean(axis=0), 'y mean:', y.mean())

## S2 — Prediction rule

The prediction rule is unchanged: `y_hat = b + X @ w`. For one row, `y_hat_i = b + x_i1*w1 + x_i2*w2`. Elastic net changes the objective used to choose the parameters, not the meaning of a sample or feature column.

## S3 — Residuals and two penalty roles

`r = y - y_hat`, `RSS = sum(r**2)`, and `MSE = RSS/N`. L1 uses `lambda_1*sum(abs(w))` and can create exact zeros. L2 uses `lambda_2*sum(w**2)` and shrinks magnitudes. The two-lambda form is primary; `(alpha, l1_ratio)` is the library parameterisation.

## S4 — The frozen objective and mapping

The objective is

```text
J = (1/(2N))*RSS + lambda_1*||w||_1 + lambda_2*||w||_2^2
  = 0.5*MSE + lambda_1*||w||_1 + lambda_2*||w||_2^2.
```

The factor `1/2` is a convention that aligns the update with scikit-learn. The exact conversion is `lambda_1 = alpha*l1_ratio`, `lambda_2 = 0.5*alpha*(1-l1_ratio)`, so `alpha = lambda_1 + 2*lambda_2` and `l1_ratio = lambda_1/(lambda_1 + 2*lambda_2)` when `alpha > 0`.

In [ ]:
def soft_threshold(z, gamma):
    return np.sign(z) * np.maximum(np.abs(z) - gamma, 0.0)

def elastic_net_coordinate_descent(X, y, lambda_1, lambda_2,
                                   max_iter=200, tol=1e-10):
    if X.ndim != 2 or X.shape[0] == 0 or X.shape[1] == 0:
        raise ValueError('X must be a non-empty 2-D array with features')
    if y.ndim != 1 or y.size == 0 or X.shape[0] != y.size:
        raise ValueError('y must be 1-D and match X sample count')
    if not np.all(np.isfinite(X)) or not np.all(np.isfinite(y)):
        raise ValueError('X and y must be finite')
    if lambda_1 < 0 or lambda_2 < 0 or max_iter < 1 or tol <= 0:
        raise ValueError('invalid penalty or convergence controls')
    b = np.float64(0.0)
    w = np.zeros(X.shape[1], dtype=np.float64)
    for _ in range(max_iter):
        old_w = w.copy()
        b = np.mean(y - X @ w)
        for j in range(X.shape[1]):
            partial = y - b - X @ w + X[:, j] * w[j]
            rho = np.mean(X[:, j] * partial)
            z = np.mean(X[:, j] * X[:, j])
            w[j] = soft_threshold(rho, lambda_1) / (z + 2.0 * lambda_2)
        if np.max(np.abs(w - old_w)) < tol:
            break
    return np.r_[b, w].astype(np.float64)

def bookkeeping(theta, lambda_1, lambda_2):
    predictions = theta[0] + X @ theta[1:]
    residuals = y - predictions
    rss = np.sum(residuals**2)
    mse = rss / y.size
    l1 = lambda_1 * np.sum(np.abs(theta[1:]))
    l2 = lambda_2 * np.sum(theta[1:]**2)
    return predictions, residuals, rss, mse, 0.5*mse, l1, l2, 0.5*mse + l1 + l2

## S5 — Coefficients and bookkeeping

The quantities below are distinct: coefficients and intercept are parameters; `lambda_1`, `lambda_2` (or `alpha`, `l1_ratio`) are hyperparameters; MSE is a reported training metric; `J` is the objective minimised by the solver.

In [ ]:
settings = {
    'OLS': (0.0, 0.0),
    'Ridge': (0.0, 0.3),
    'Lasso': (0.8, 0.0),
    'ENet-A': (0.2, 0.3),
    'ENet-B': (0.8, 0.1),
}
for name, (lambda_1, lambda_2) in settings.items():
    theta = elastic_net_coordinate_descent(X, y, lambda_1, lambda_2)
    *_, mse, half_mse, l1, l2, objective = bookkeeping(theta, lambda_1, lambda_2)
    print(f'{name:7s} theta={theta}, MSE={mse:.3f}, L1={l1:.3f}, L2={l2:.3f}, J={objective:.3f}')

## S6 — The modified coordinate update

For `rho_j = (1/N)*x_j.T@r_j` and `z_j = (1/N)*x_j.T@x_j`, the update is

```text
w_j <- S(rho_j, lambda_1) / (z_j + 2*lambda_2).
```

The L2 term adds curvature in the denominator. The numerator still controls the exact-zero threshold.

In [ ]:
Z = X - X.mean(axis=0)
yc = y - y.mean()
print('Z.T @ Z =\n', Z.T @ Z)
print('Z.T @ yc =', Z.T @ yc)
print('rho_c =', (Z.T @ yc) / y.size, 'z_c =', np.diag(Z.T @ Z) / y.size)
print('closed forms:')
for name, (lambda_1, lambda_2) in settings.items():
    w = soft_threshold(np.array([1.4, -0.4]), lambda_1) / (0.4 + 2*lambda_2)
    print(f'{name:7s} b={4.2-w.sum():.3f}, w={w}')

## S8 — Deterministic implementation and verification

The implementation uses `np.float64`, validates dimensions and finite values, and keeps the intercept unpenalised. The independent verifier recomputes the closed forms, objective components, and KKT conditions without calling this notebook's solver.

In [ ]:
from sklearn.linear_model import ElasticNet
for name, (lambda_1, lambda_2) in settings.items():
    if name == 'OLS':
        continue
    alpha = lambda_1 + 2*lambda_2
    l1_ratio = lambda_1 / alpha
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio,
                       fit_intercept=True, max_iter=100_000, tol=1e-10)
    model.fit(X, y)
    print(name, np.r_[model.intercept_, model.coef_])

print('For the pure-L2 reference, Ridge(alpha=2*N*lambda_2) means Ridge(alpha=3.0).')

## S10 — Visual and correlated-feature boundary

The static figure compares coefficients, illustrative L1/L2/elastic-net contours, and the fixed `alpha=0.8` mixing scan. Direct labels, markers, and hatching keep meaning independent of colour. The duplicated-feature example is an extreme synthetic illustration of a tendency, not a guarantee for merely correlated features.

![Three-panel elastic-net visual: coefficient comparison, illustrative penalty geometry, and fixed-strength mixing scan.](assets/elastic_net_panels.png)

## S11 — Failure modes and exercises

Watch for swapped sample/feature axes, non-finite data, wrong feature counts, the factor-of-two mapping trap, penalising the intercept, and confusing `J` with MSE. A zero coefficient is not proof of irrelevance or causality.

Exercises: evaluate `J` at a non-optimal parameter vector; convert lambda pairs to alpha/ratio and back; predict zeros at `(0.4, 0.2)`, `(0.9, 0.1)`, and `(1.5, 0.1)`; trace one cycle at `(0.8, 0.2)`; and explain the duplicated-feature lasso split.

## S12 — Takeaways and continuity

Elastic net minimises `0.5*MSE` plus L1 and L2 penalties. L1 supplies the threshold; L2 adds denominator shrinkage. The two parameterisations are exactly related, and RSS, MSE, penalties, and `J` remain distinct. All results here are deterministic arithmetic on a tiny synthetic table.

References: Zou & Hastie (2005), Friedman, Hastie & Tibshirani (2010), the glmnet vignette, scikit-learn 1.5 ElasticNet documentation, Hoerl & Kennard (1970), Tibshirani (1996), and NumPy 1.26 documentation.